# X1 - Transfer test (drift check, RQ3)

**Run this notebook locally, not on Colab**: it uses your team's consented own logs, which never leave the machine.

1. Collect own logs (Stage 2 `run_own_logs`) and clean + redact them (Stage 3); skim the redacted segments.
2. Export a sheet of ~150 own-log segments, label it by hand, import it.
3. Re-run Stage 6 so the labels go into the transfer split.
4. Run X1 **once** with the current registered model. A second run is refused unless forced with a written reason, which is recorded.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.evaluate import transfer

transfer.export_transfer_sheet(150)   # -> data/labels/transfer_sheet.csv; label it like the gold sheets

In [ ]:
n, problems = transfer.import_transfer_sheet(rater_id="rater1")
n, problems   # fix any listed rows in the sheet, then re-import

In [ ]:
from krs.split import make_splits

make_splits("split")["transfer"]   # transfer split now filled

## Run X1 (once)

In [ ]:
rep = transfer.run("runs/X1")
{k: rep[k] for k in ("model_version", "transfer_segments", "transfer_conversations", "gold_agentic_segments")}

In [ ]:
import pandas as pd

rows = []
for axis in ("knowledge_types", "domains"):
    a = rep[axis]
    rows.append({"axis": axis, "transfer": a["transfer"]["macro_f1"], "gold (all)": a["gold"]["macro_f1"],
                 "gold (agentic)": a.get("gold_agentic", {}).get("macro_f1"),
                 "drop vs gold": a["drop_gold_minus_transfer"]["diff"],
                 "drop 95% CI": f"{a['drop_gold_minus_transfer']['ci_low']:+.3f} to {a['drop_gold_minus_transfer']['ci_high']:+.3f}",
                 "drop vs agentic gold": a.get("drop_gold_agentic_minus_transfer", {}).get("diff")})
pd.DataFrame(rows)

## Error analysis: where does it break? (per label, and how label frequencies shifted)

In [ ]:
ax = "knowledge_types"
pd.DataFrame({"F1 transfer": {r["label"]: r["f1"] for r in rep[ax]["per_label_transfer"]},
              "F1 gold": {r["label"]: r["f1"] for r in rep[ax]["per_label_gold"]},
              "share transfer": rep[ax]["label_share_transfer"], "share gold": rep[ax]["label_share_gold"]})

In [ ]:
ax = "domains"
pd.DataFrame({"F1 transfer": {r["label"]: r["f1"] for r in rep[ax]["per_label_transfer"]},
              "F1 gold": {r["label"]: r["f1"] for r in rep[ax]["per_label_gold"]},
              "share transfer": rep[ax]["label_share_transfer"], "share gold": rep[ax]["label_share_gold"]})

In [ ]:
rep["knowledge_bearing_f1"], rep["length_chars_median"]